# Factorial NHMM — independent regime dimensions

Many real systems have **multiple independent regime dimensions** that don't reduce to one. A market has *trend* (up / sideways / down) **and** *volatility* (low / high) — but they're driven by different forces and don't always co-move.

Modelling this as a flat 6-state joint HMM forces the model to enumerate every (trend, vol) combination as its own state and learn the full 6×6 transition matrix from scratch. **Factorial NHMM** keeps the chains separate : two transition matrices (3×3 for trend, 2×2 for vol) with per-chain covariates.

**This notebook** : 2 chains (trend × vol), each driven by its own covariate, fit with `fit_factorial_nhmm` and inspected per-chain.

## 1. Simulate trend × vol with independent dynamics

Trend chain : 3 states (down, flat, up), driven by `macro_index`. Vol chain : 2 states (low, high), driven by `fear_index`. The observations are 2-D Gaussian whose mean depends on the joint (trend, vol) state.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
T = 1200

macro_index = rng.normal(0, 1, (T, 1))   # drives trend transitions
fear_index = rng.normal(0, 1, (T, 1))    # drives vol transitions

# Joint state means : (trend, vol) -> 2-D mean. Flattened row-major (trend × vol).
joint_means = np.array([
    [-2.0, -2.0],   # (down, low)
    [-2.0,  2.0],   # (down, high)
    [ 0.0, -1.0],   # (flat, low)
    [ 0.0,  1.0],   # (flat, high)
    [ 2.0, -2.0],   # (up,   low)
    [ 2.0,  2.0],   # (up,   high)
])
cov = 0.4 * np.eye(2)

trend, vol = 1, 0   # start (flat, low)
X = np.zeros((T, 2))
true_trend = np.zeros(T, dtype=int)
true_vol = np.zeros(T, dtype=int)
for t in range(T):
    joint = trend * 2 + vol
    X[t] = rng.multivariate_normal(joint_means[joint], cov)
    true_trend[t], true_vol[t] = trend, vol
    # Trend dynamics : macro_index biases toward up; sticky otherwise
    if rng.random() < 0.85:
        pass
    else:
        trend = int(np.clip(trend + (1 if macro_index[t, 0] > 0.3 else -1 if macro_index[t, 0] < -0.3 else 0), 0, 2))
    # Vol dynamics : fear_index biases toward high; less sticky than trend
    if rng.random() < 0.80:
        pass
    else:
        vol = 1 if fear_index[t, 0] > 0 else 0

print(f"T = {T}")
print(f"Trend distribution : {np.bincount(true_trend) / T}")
print(f"Vol distribution   : {np.bincount(true_vol) / T}")

## 2. Declare chains and fit

Each chain has its own `FactorialChainSpec(name, n_states)`. Per-chain covariates are passed in a dict keyed by chain name. The joint emission is Gaussian over the 6-state product space.

In [ ]:
from hmm_core.factorial_nhmm import FactorialChainSpec, fit_factorial_nhmm
from hmm_core.topology import EmissionSpec, FitSpec, InitSpec

chains = [
    FactorialChainSpec(name="trend", n_states=3),
    FactorialChainSpec(name="vol",   n_states=2),
]

result = fit_factorial_nhmm(
    chains,
    X,
    covariates_per_chain={"trend": macro_index, "vol": fear_index},
    covariate_names_per_chain={"trend": ["macro_index"], "vol": ["fear_index"]},
    emission=EmissionSpec(type="gaussian", covariance_type="diag", n_features=2),
    fit_spec=FitSpec(algorithm="baum_welch", n_iter=50, tol=1e-3),
    init=InitSpec(strategy="kmeans", seed=42),
    seed=42,
)
result

## 3. Parameter savings vs joint HMM

The whole point : transition parameters factor by chain. For D chains of K_d states each :
- **Joint HMM** : K_joint² = (∏K_d)² parameters in the transition matrix.
- **Factorial NHMM** : Σ_d K_d² (per chain) — way fewer.

In [ ]:
K_per_chain = result.K_per_chain
K_joint = result.K_joint

joint_params = K_joint ** 2
factorial_params = sum(k ** 2 for k in K_per_chain)
savings = joint_params / factorial_params

print(f"K_per_chain = {K_per_chain}, K_joint = {K_joint}")
print(f"Joint HMM transition params      : {joint_params}")
print(f"Factorial NHMM transition params : {factorial_params}")
print(f"Savings : {savings:.1f}×")

## 4. Per-chain decoded paths

`decode_chain(X, chain_name)` projects the joint Viterbi to one chain. Each chain decodes independently from the same observation sequence.

In [ ]:
decoded_trend = result.decode_chain(X, "trend")
decoded_vol = result.decode_chain(X, "vol")

# Try all label permutations and report the best alignment accuracy
from itertools import permutations

def best_accuracy(pred, true, K):
    return max(
        np.mean(np.array([perm[p] for p in pred]) == true)
        for perm in permutations(range(K))
    )

print(f"Trend chain accuracy (best perm) : {best_accuracy(decoded_trend, true_trend, 3):.2%}")
print(f"Vol chain accuracy   (best perm) : {best_accuracy(decoded_vol, true_vol, 2):.2%}")

## 5. Inspect each chain's A_t

`A_t(chain_name)` returns the (T, K_d, K_d) array of time-varying transitions for that chain. Each chain's transitions are modulated by **its own** covariate, not by the other chain's.

In [ ]:
A_trend = result.A_t("trend")
A_vol = result.A_t("vol")

print(f"trend A_t shape : {A_trend.shape}   (T, K, K)")
print(f"vol   A_t shape : {A_vol.shape}")
print()
print("trend transition matrix averaged over t :")
print(A_trend.mean(axis=0).round(3))
print()
print("vol transition matrix averaged over t :")
print(A_vol.mean(axis=0).round(3))

## Next

- **Textbook canonicals** : the smoothing / Viterbi machinery validated against AIMA and Durbin in notebooks 06 and 07.
- **Design rationale** : the 2-stage decomposition (rejected joint-logit expansion) is documented in `docs/specs/2026-05-22-phase-a13-factorial-nhmm.md`.